# 01 - Explore: wav2vec2-th Embeddings

สำรวจ embedding ที่สกัดได้จากโมเดล `airesearch/wav2vec2-large-xlsr-53-th` (ผ่านสคริปต์ `src/preprocessing/extract_embedding.py`) ซึ่งบันทึกไว้เป็นไฟล์ `.npy` แยกตาม layer (layer 0 = ผลลัพธ์จาก CNN feature encoder, layer 1-24 = ผลลัพธ์จาก transformer block แต่ละชั้น) อยู่ในโฟลเดอร์ `data/embeddings/`

In [ ]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

EMBEDDINGS_DIR = os.path.join("..", "data", "embeddings")

## 1. โหลด embeddings

หา utterance แรกที่เจอในโฟลเดอร์ `data/embeddings/` โดยอัตโนมัติ (อิงจากชื่อไฟล์ `{utterance_id}_layer00.npy`) แล้วโหลด `.npy` ของทุก layer มาดู พร้อมพิมพ์ shape ของแต่ละ layer

In [ ]:
# หา utterance แรกโดยอัตโนมัติ จากไฟล์ layer00 ตัวแรกที่เจอใน data/embeddings/
layer00_files = sorted(glob.glob(os.path.join(EMBEDDINGS_DIR, "*_layer00.npy")))
assert layer00_files, f"ไม่พบไฟล์ embedding ใน {EMBEDDINGS_DIR}"

utterance_id = os.path.basename(layer00_files[0])[: -len("_layer00.npy")]
layer_files = sorted(
    glob.glob(os.path.join(EMBEDDINGS_DIR, f"{utterance_id}_layer*.npy"))
)
num_layers = len(layer_files)

print(f"Utterance ที่เลือก: {utterance_id}")
print(f"จำนวน layer ที่พบ: {num_layers}")

In [ ]:
layers = []
for layer_idx in range(num_layers):
    path = os.path.join(
        EMBEDDINGS_DIR, f"{utterance_id}_layer{layer_idx:02d}.npy"
    )
    layer_array = np.load(path)
    layers.append(layer_array)
    print(f"Layer {layer_idx:02d} shape: {layer_array.shape}")

## 2. Heatmap visualization

เปรียบเทียบ heatmap ของ layer 0 (CNN output), layer 12 (transformer ชั้นกลาง) และ layer 24 (transformer ชั้นสุดท้าย) — แกน X คือ time steps, แกน Y คือ hidden dimensions

In [ ]:
layer_indices_to_plot = [0, 12, num_layers - 1]
layer_titles = [
    "Layer 0 - CNN feature encoder output",
    "Layer 12 - middle transformer block",
    f"Layer {num_layers - 1} - last transformer block",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, layer_idx, title in zip(axes, layer_indices_to_plot, layer_titles):
    embedding = layers[layer_idx]  # shape: (time_steps, hidden_dim)
    im = ax.imshow(embedding.T, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Time steps")
    ax.set_ylabel("Hidden dimension")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Activation")

fig.suptitle(
    f"wav2vec2-th layer activations - utterance {utterance_id.split('_')[0]}",
    y=1.03,
)
fig.tight_layout()
plt.show()

## 3. Layer comparison

พล็อตค่าเฉลี่ย activation ของแต่ละ layer (0-24) เป็นกราฟเส้น เพื่อดูว่าข้อมูล เปลี่ยนแปลงไปอย่างไรเมื่อผ่านแต่ละ layer ของโมเดล

In [ ]:
mean_activations = [float(layer.mean()) for layer in layers]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    range(num_layers),
    mean_activations,
    color="#2A6F97",
    linewidth=2,
    marker="o",
    markersize=4,
)
ax.set_xlabel("Layer index")
ax.set_ylabel("Mean activation value")
ax.set_title(
    "Mean activation across wav2vec2-th layers "
    "(0 = CNN output, 24 = last transformer block)"
)
ax.set_xticks(range(0, num_layers, 2))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. เลือก layer ไหนดี?

จากงานวิจัยเกี่ยวกับ wav2vec2 พบว่า **layer ชั้นกลาง (ประมาณ layer 6-12)** มักให้ representation ที่เหมาะกับข้อมูลเชิง **เนื้อหา/สัทศาสตร์ (content / phonetic information)** มากกว่า layer สุดท้าย (layer 24) เพราะ:

- layer ต้น ๆ (ใกล้ CNN output) ยังมีข้อมูล acoustic/spectral แบบดิบอยู่มาก
- layer ชั้นกลางเริ่มเข้ารหัสข้อมูลระดับหน่วยเสียง (phoneme-level information) ได้ดี
- layer ท้าย ๆ ถูกปรับให้เหมาะกับ task ปลายทางของโมเดล (เช่น CTC/ASR) มากกว่าจะเป็น general-purpose representation จึงอาจสูญเสียรายละเอียดเชิงสัทศาสตร์บางส่วนไป

**ยังไม่ต้องฟันธงว่าจะใช้ layer ไหนในโปรเจกต์นี้ — เราจะทดลองเปรียบเทียบและตัดสินใจเลือก layer ที่เหมาะสมที่สุดใน Week 3**